# Imports

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import re

from src import gradio_utils

from src.parse_labeling import parse_labeled_file
from src.wikidata_utils import translate_term, wikidata_gradio

# Read File

## Concat ARC

## Claude Version

## GSM8K

In [ ]:
file_name = f'labeled_files/gsm8k_comparison_200.txt'
csv_name = f'manual_compare/gsm8k_comparison_200_FULL.csv'
old_label_df, new_df = parse_labeled_file(file_name, csv_name)

old_label_df.reset_index(inplace=True, drop=True)
new_df.reset_index(inplace=True, drop=True)

print((new_df['option 1'] == old_label_df['option 1']).sum())
print((new_df['option 2'] == old_label_df['option 2']).sum())

old_label_df.shape, new_df.shape

In [ ]:
new_df.index, old_label_df.index

In [ ]:
display(old_label_df.head(2))
display(new_df.head(2))

## MMLU

In [22]:
new_df = pd.read_csv('manual_compare/mmlu_test_each_90_FULL.csv')
print(new_df.shape)
new_df.head(2)

(157, 5)


,original,model 1,model 2,option 1,option 2
0,<question>Find the degree for the given field ...,claude_thinking_v2,gemini_pro_think_v1,<question>מצאו את המעלה עבור הרחבת השדה הנתונה...,<question>מצאו את המעלה עבור הרחבת השדה הנתונה...
1,<question>Statement 1 | If a finite group has ...,gemini_pro_think_v1,claude_thinking_v2,"<question>טענה 1 | אם לחבורה סופית יש סדר n, א...",<question>היגד 1 | אם חבורה סופית היא מסדר n א...


In [23]:
# Concat the train which was labeled
mmlu_train = pd.read_csv('labeled_files/mmlu_train_labeled_gradio.csv')
mmlu_train.fillna('', inplace=True)
mmlu_train = mmlu_train[(mmlu_train['gold'] != '') | (mmlu_train['rating'] != '')]
print(mmlu_train.shape)
mmlu_train.head(2)

(20, 6)


,text_column,new_text_column,gold,rating,model 1,model 2
0,<question>The cyclic subgroup of Z_24 generate...,Option 1:\n<question>הסדר של התת-חבורה הציקלית...,,1,gemini_pro_think_v1,claude_v1_refine
1,<question>Statement 1 | The symmetric group S_...,Option 1:\n<question>טענה 1 | החבורה הסימטרית ...,,1,gemini_pro_think_v1,claude_v1_refine


## ARC only gemini

In [102]:
new_df = pd.read_csv('compare_csv/arc_ai2_chall_test_503-853.csv')
print(new_df.shape)
new_df = new_df.tail(149 + 51).reset_index(drop=True)
print(new_df.shape)
new_df.head(2)

(349, 2)
(200, 2)


,original,gemini_pro_think_v2
0,<question>Many bacteria grow quickly at room t...,<question>חיידקים רבים גדלים במהירות בטמפרטורת...
1,<question>Some plants grow in an area where th...,<question>צמחים מסוימים גדלים באזור שיש בו בעל...


In [103]:
arc_gemini_df = pd.DataFrame()
arc_gemini_df['text_column'] = new_df['original']
arc_gemini_df['new_text_column'] = new_df['gemini_pro_think_v2']
arc_gemini_df['gold'] = ''
arc_gemini_df['rating'] = ''
print(arc_gemini_df.shape)
arc_gemini_df.head(2)

(200, 4)


,text_column,new_text_column,gold,rating
0,<question>Many bacteria grow quickly at room t...,<question>חיידקים רבים גדלים במהירות בטמפרטורת...,,
1,<question>Some plants grow in an area where th...,<question>צמחים מסוימים גדלים באזור שיש בו בעל...,,


In [ ]:
old_test_arc = pd.read_csv('labeled_files/arc_ai_TEST_labeled_gradio.csv')
old_test_arc = old_test_arc.fillna('')
old_test_arc['rating'] = old_test_arc['rating'].apply(str).replace('1.0', '1').replace('2.0', '2')

arc_gemini_df = pd.concat([old_test_arc, arc_gemini_df], ignore_index=True)
arc_gemini_df = arc_gemini_df.fillna('')
arc_gemini_df.shape

In [ ]:
display(arc_gemini_df.head())
display(arc_gemini_df.tail())

In [105]:
arc_gemini_df.to_csv('labeled_files/arc_ai_TEST_2_labeled_gradio.csv', index=False)

## GSM only gemini

### For Israel

In [14]:
# new_df = pd.read_csv('compare_csv/gsm8k_main_test_201-500.csv')
new_df = pd.read_csv('compare_csv/gsm8k_main_test_501-600.csv')
new_df_2 = pd.read_csv('compare_csv/gsm8k_main_test_601-1000.csv').head(200)
new_df = pd.concat([new_df, new_df_2], ignore_index=True)
print(new_df.shape)
new_df.head(2)

(300, 2)


,original,gemini_pro_think_v2
0,"<question>Together Lily, David, and Bodhi coll...","<question>ביחד, לילי, דוד ובועז אספו 43 חרקים...."
1,<question>Mariah’s grandma was teaching her to...,<question>סבתא של מריה לימדה אותה לסרוג. מריה ...


In [15]:
gsm_gemini_df = pd.DataFrame()
gsm_gemini_df['text_column'] = new_df['original']
gsm_gemini_df['new_text_column'] = new_df['gemini_pro_think_v2']
gsm_gemini_df['gold'] = ''
gsm_gemini_df['rating'] = ''
print(gsm_gemini_df.shape)
gsm_gemini_df.head(2)

(300, 4)


,text_column,new_text_column,gold,rating
0,"<question>Together Lily, David, and Bodhi coll...","<question>ביחד, לילי, דוד ובועז אספו 43 חרקים....",,
1,<question>Mariah’s grandma was teaching her to...,<question>סבתא של מריה לימדה אותה לסרוג. מריה ...,,


In [16]:
old_test_gsm = pd.read_csv('labeled_files/gsm_TEST_labeled_gradio.csv')
old_test_gsm = old_test_gsm.fillna('')
old_test_gsm['rating'] = old_test_gsm['rating'].apply(str).replace('1.0', '1').replace('2.0', '2')

gsm_gemini_df = pd.concat([old_test_gsm, gsm_gemini_df], ignore_index=True)
gsm_gemini_df = gsm_gemini_df.fillna('')
gsm_gemini_df.shape

(600, 16)

In [19]:
display(gsm_gemini_df.head(2))
display(gsm_gemini_df.iloc[300:302])
display(gsm_gemini_df.tail(2))

,text_column,new_text_column,gold,rating,category_annotation_1,severity_annotation_1,category_annotation_2,severity_annotation_2,category_annotation_3,severity_annotation_3,category_annotation_4,severity_annotation_4,category_annotation_5,severity_annotation_5,category_annotation_6,severity_annotation_6
0,<question>Baldur gets water from a well. He ge...,<question>בלדור שואב מים מבאר. הוא שואב 5 דליי...,<question>ולדי שואב מים מבאר. הוא שואב 5 דליי ...,1,,,,,,,,,,,,
1,<question>John wins an award at work. The awa...,<question>יוני זוכה בפרס בעבודה. הפרס כולל מענ...,,,,,,,,,,,,,,


,text_column,new_text_column,gold,rating,category_annotation_1,severity_annotation_1,category_annotation_2,severity_annotation_2,category_annotation_3,severity_annotation_3,category_annotation_4,severity_annotation_4,category_annotation_5,severity_annotation_5,category_annotation_6,severity_annotation_6
300,"<question>Together Lily, David, and Bodhi coll...","<question>ביחד, לילי, דוד ובועז אספו 43 חרקים....",,,,,,,,,,,,,,
301,<question>Mariah’s grandma was teaching her to...,<question>סבתא של מריה לימדה אותה לסרוג. מריה ...,,,,,,,,,,,,,,


,text_column,new_text_column,gold,rating,category_annotation_1,severity_annotation_1,category_annotation_2,severity_annotation_2,category_annotation_3,severity_annotation_3,category_annotation_4,severity_annotation_4,category_annotation_5,severity_annotation_5,category_annotation_6,severity_annotation_6
598,"<question>When Billy was first hired, he was p...","<question>כשבני התקבל לעבודה לראשונה, הוא קיבל...",,,,,,,,,,,,,,
599,<question>A loaf of bread at the bakery costs ...,<question>כיכר לחם במאפייה עולה 2 שקלים. בייגל...,,,,,,,,,,,,,,


In [20]:
gsm_gemini_df.to_csv('labeled_files/gsm_TEST_labeled_gradio.csv', index=False)

### For Guy

In [23]:
# overlap_df = pd.read_csv('compare_csv/gsm8k_main_test_201-500.csv').tail(70)
new_df = pd.read_csv('compare_csv/gsm8k_main_test_601-1000.csv').tail(250).reset_index(drop=True)

# new_df = pd.concat([overlap_df, new_df], ignore_index=True)

# new_df = pd.read_csv('compare_csv/gsm8k_main_test_501-600.csv')
print(new_df.shape)
display(new_df.head(2))
display(new_df.tail(2))

(250, 2)


,original,gemini_pro_think_v2
0,<question>Candy has a chair rental business. D...,<question>לקנדי יש עסק להשכרת כיסאות. במהלך ימ...
1,"<question>Gunther, the gorilla, had 48 bananas...","<question>לגונתר, הגורילה, היו 48 בננות מוחבאו..."


,original,gemini_pro_think_v2
248,<question>Dylan attended a wedding where there...,<question>דילן השתתף בחתונה שבה היו 100 אורחים...
249,<question>A family of 6 (2 adults and 4 kids) ...,<question>משפחה בת 6 נפשות (2 מבוגרים ו-4 ילדי...


In [24]:
gsm_gemini_df = pd.DataFrame()
gsm_gemini_df['text_column'] = new_df['original']
gsm_gemini_df['new_text_column'] = new_df['gemini_pro_think_v2']
gsm_gemini_df['gold'] = ''
gsm_gemini_df['rating'] = ''
print(gsm_gemini_df.shape)
gsm_gemini_df.head(2)

(250, 4)


,text_column,new_text_column,gold,rating
0,<question>Candy has a chair rental business. D...,<question>לקנדי יש עסק להשכרת כיסאות. במהלך ימ...,,
1,"<question>Gunther, the gorilla, had 48 bananas...","<question>לגונתר, הגורילה, היו 48 בננות מוחבאו...",,


In [25]:
display(gsm_gemini_df.head(3))
display(gsm_gemini_df.tail(3))

,text_column,new_text_column,gold,rating
0,<question>Candy has a chair rental business. D...,<question>לקנדי יש עסק להשכרת כיסאות. במהלך ימ...,,
1,"<question>Gunther, the gorilla, had 48 bananas...","<question>לגונתר, הגורילה, היו 48 בננות מוחבאו...",,
2,<question>Jenna has 4 roommates. Each month th...,<question>ליעל יש 4 שותפות לדירה. בכל חודש חשב...,,


,text_column,new_text_column,gold,rating
247,<question>Walter is collecting money for chari...,<question>איתן אוסף כסף לצדקה. תחילה הוא אוסף ...,,
248,<question>Dylan attended a wedding where there...,<question>דילן השתתף בחתונה שבה היו 100 אורחים...,,
249,<question>A family of 6 (2 adults and 4 kids) ...,<question>משפחה בת 6 נפשות (2 מבוגרים ו-4 ילדי...,,


In [26]:
gsm_gemini_df.to_csv('labeled_files/gsm_TEST_2_labeled_gradio.csv', index=False)

## MMLU only Gemini

In [2]:
new_df_1 = pd.read_csv('compare_csv/mmlu_test_main_sub_300.csv')
new_df_2 = pd.read_csv('compare_csv/mmlu_test_main_sub_301-500.csv')

new_df = pd.concat([new_df_1, new_df_2], ignore_index=True)

print(new_df.shape)
new_df.head(2)

(498, 2)


,original,gemini_pro_think_v1
0,"<question>Let p = (1, 2, 5, 4)(2, 3) in S_5 . ...","<question>תהי p = (1, 2, 5, 4)(2, 3) ב-S_5. מצ..."
1,<question>Find all zeros in the indicated fini...,<question>מצאו את כל האפסים בשדה הסופי המצוין ...


In [3]:
mmlu_gemini_df = pd.DataFrame()
mmlu_gemini_df['text_column'] = new_df['original']
mmlu_gemini_df['new_text_column'] = new_df['gemini_pro_think_v1']
mmlu_gemini_df['gold'] = ''
mmlu_gemini_df['rating'] = ''
print(mmlu_gemini_df.shape)
mmlu_gemini_df.head(2)

(498, 4)


,text_column,new_text_column,gold,rating
0,"<question>Let p = (1, 2, 5, 4)(2, 3) in S_5 . ...","<question>תהי p = (1, 2, 5, 4)(2, 3) ב-S_5. מצ...",,
1,<question>Find all zeros in the indicated fini...,<question>מצאו את כל האפסים בשדה הסופי המצוין ...,,


In [ ]:
display(mmlu_gemini_df.head())
display(mmlu_gemini_df.tail())

In [5]:
mmlu_gemini_df.to_csv('labeled_files/mmlu_main_sub_TEST_labeled_gradio.csv', index=False)

# Create UI

## Create new file

## Create overlap file

In [3]:
gsm_save_to = 'labeled_files/gsm8k_labeled_gradio.csv'
arc_save_to = 'labeled_files/arc_ai_labeled_gradio.csv'
mmlu_save_to = 'labeled_files/mmlu_test_labeled_gradio.csv'

# overlap_save_to = 'labeled_files/overlap_gradio.csv'

def read_and_fill(file_path):
    df = pd.read_csv(file_path)
    return df.fillna('')
    
gsm_labeled_df = read_and_fill(gsm_save_to)
arc_labeled_df = read_and_fill(arc_save_to)
mmlu_labeled_df = read_and_fill(mmlu_save_to)

## Run existing file

___

In [8]:
mmlu_save_to = 'labeled_files/mmlu_main_sub_TEST_labeled_gradio.csv'

In [9]:
mmlu_labeled_df = pd.read_csv(mmlu_save_to)
mmlu_labeled_df = mmlu_labeled_df.fillna('')
mmlu_labeled_df['rating'] = mmlu_labeled_df['rating'].apply(str).replace('1.0', '1').replace('2.0', '2')

In [10]:
mmlu_labeled_df.shape

(498, 4)

In [11]:
display(mmlu_labeled_df.head(2))
display(mmlu_labeled_df.tail(2))

,text_column,new_text_column,gold,rating
0,"<question>Let p = (1, 2, 5, 4)(2, 3) in S_5 . ...","<question>תהי p = (1, 2, 5, 4)(2, 3) ב-S_5. מצ...",,
1,<question>Find all zeros in the indicated fini...,<question>מצאו את כל האפסים בשדה הסופי המצוין ...,,


,text_column,new_text_column,gold,rating
496,<question>Delusions of grandeur are most chara...,<question>מחשבות שווא של גדלות הן המאפיין הבול...,,
497,<question>Abraham Maslow proposed the idea tha...,<question>אברהם מאסלו הציע את הרעיון שחלק מהמנ...,,


In [12]:
mmlu_labeled_df['rating'].dtype

dtype('O')

In [13]:
# This is the MMLU server
mmlu_server = gradio_utils.run_annotator(mmlu_labeled_df, mmlu_save_to, ' - MMLU Test benchamrk - Gemini only')
wiki_server = wikidata_gradio()

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b923327f44821f0efb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://64a7007408c231597d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
mmlu_server.close()

In [ ]:
wiki_server.close()

___

In [2]:
gsm_save_to = 'labeled_files/gsm_TEST_labeled_gradio.csv'

In [3]:
gsm_labeled_df = pd.read_csv(gsm_save_to)
gsm_labeled_df = gsm_labeled_df.fillna('')
gsm_labeled_df['rating'] = gsm_labeled_df['rating'].apply(str).replace('1.0', '1').replace('2.0', '2')

In [4]:
gsm_labeled_df.shape

(600, 16)

In [ ]:
display(gsm_labeled_df.head(2))
display(gsm_labeled_df.tail(2))

In [6]:
gsm_labeled_df['rating'].dtype

dtype('O')

In [7]:
# This is the GSM8K server
gsm_server = gradio_utils.run_annotator(gsm_labeled_df, gsm_save_to, ' - GSM8K Test benchamrk - Gemini only')
wiki_server = wikidata_gradio()

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2143aad9bbd6829ed6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://fc53ecb3bef5c0018d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
gsm_server.close()

In [ ]:
wiki_server.close()

___

In [8]:
# second_labeler_save_to = 'labeled_files/overlap_gradio.csv'
second_labeler_save_to = 'labeled_files/arc_ai_TEST_2_labeled_gradio.csv'

In [9]:
second_labeled_df = pd.read_csv(second_labeler_save_to)
second_labeled_df = second_labeled_df.fillna('')
second_labeled_df['rating'] = second_labeled_df['rating'].apply(str).replace('1.0', '1').replace('2.0', '2')

In [10]:
second_labeled_df['rating'].dtype

dtype('O')

In [11]:
# Different servers
second_labeler_server = gradio_utils.run_annotator(second_labeled_df, second_labeler_save_to, 'ARC-AI2 second- ')
second_wiki_server = wikidata_gradio()

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://f176081f9de405672b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://2374ffd01cbfde990e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
second_labeler_server.close()

In [ ]:
second_wiki_server.close()

___

In [12]:
gsm_second_labeler_save_to = 'labeled_files/gsm_TEST_2_labeled_gradio.csv'

In [13]:
gsm_second_labeled_df = pd.read_csv(gsm_second_labeler_save_to)
gsm_second_labeled_df = gsm_second_labeled_df.fillna('')
gsm_second_labeled_df['rating'] = gsm_second_labeled_df['rating'].apply(str).replace('1.0', '1').replace('2.0', '2')

In [14]:
# Different servers
gsm_second_labeler_server = gradio_utils.run_annotator(gsm_second_labeled_df, gsm_second_labeler_save_to, 'GSM8K second - ')

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://b81b02847474bde9bf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
second_labeler_server.close()